# 08 - Tuning Iperparametri Modelli ML
Ottimizzazione bayesiana con Optuna degli iperparametri di XGBoost e LightGBM (Sezione 3.6 della tesi).

In [ ]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

## Import

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import json
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)
import xgboost as xgb
import lightgbm as lgb

from lib.data import load_splits, rescale_data, calculate_metrics, print_results, categorize_glucose, HYPO, HYPER, L_BOUND, U_BOUND

Installazione di Optuna per l'ottimizzazione bayesiana degli iperparametri.

In [ ]:
# installa optuna
!pip install optuna

In [ ]:
import optuna
from optuna.samplers import TPESampler

## Funzioni di utilità
Definizione dello spazio di ricerca, funzione obiettivo (MAE medio per paziente sul validation set) e ottimizzazione con Optuna (TPE sampler).

In [ ]:
def calculate_patient_based_mae(df):
    """Calcola la media dei MAE calcolata per ogni paziente individualmente."""
    _, maes, _, _ = calculate_metrics(df)
    return np.mean(maes) if maes else float("inf")

In [ ]:
def create_model_instance(model_name, seed, params):
    """Costruisci un'istanza di modello con i parametri passati"""
    if model_name == "xgb":
        return xgb.XGBRegressor(**params, random_state=seed, device="cuda:0")
    elif model_name == "lgb":
        return lgb.LGBMRegressor(
            **params, random_state=seed, device="gpu", verbosity=-1
        )
    else:
        raise ValueError(f"Unsupported model: {model_name}")

In [ ]:
def evaluate_model_on_validation(model, val_set, X_cols, y_cols):
    """Valuta il modello sul validation set e restituisci il MAE medio dei pazienti del val_set."""
    eval_set = val_set.copy()
    eval_set["y_pred"] = model.predict(eval_set[X_cols])
    eval_set = eval_set.rename(columns={y_cols[-1]: "target"})
    eval_set = rescale_data(eval_set, ["target", "y_pred"])

    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = eval_set[output_columns]
    return calculate_patient_based_mae(results)

In [ ]:
def create_objective_function(model_name, train_set, val_set, X_cols, y_cols, seed):
    """Costruisci la funzuone obiettivo di Optuna per l'ottimizzazione degli iperparametri."""

    def objective(trial):
        if model_name == "xgb":
            trial_params = {
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "min_child_weight": trial.suggest_float(
                    "min_child_weight", 0.1, 10.0, log=True
                ),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "learning_rate": trial.suggest_float(
                    "learning_rate", 0.01, 0.3, log=True
                ),
                "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
                "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
            }
        elif model_name == "lgb":
            trial_params = {
                "num_leaves": trial.suggest_int("num_leaves", 10, 300),
                "max_depth": trial.suggest_int("max_depth", 3, 12),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                "min_child_weight": trial.suggest_float(
                    "min_child_weight", 1e-3, 10.0, log=True
                ),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "learning_rate": trial.suggest_float(
                    "learning_rate", 0.01, 0.3, log=True
                ),
                "n_estimators": trial.suggest_int("n_estimators", 50, 300),
                "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 50),
            }

        # Addestra il modello sul train set
        model = create_model_instance(model_name, seed, trial_params)
        model.fit(train_set[X_cols], train_set[y_cols[-1]])

        # Valuta sul validation set
        mae = evaluate_model_on_validation(model, val_set, X_cols, y_cols)
        return mae

    return objective

In [ ]:
def run_optimization(model_name, train_set, val_set, X_cols, y_cols, args):
    """Esegui l'ottimizzazione con Optuna per uno specifico modello."""
    print(f"\n{'='*60}")
    print(f"HYPERPARAMETER TUNING - {model_name.upper()}")
    print(f"{'='*60}")

    # Valuta le performance del modello "baseline"
    print(f"Evaluating baseline {model_name.upper()} model...")
    baseline_model = create_model_instance(model_name, args.seed, {})
    baseline_model.fit(train_set[X_cols], train_set[y_cols[-1]])
    baseline_mae = evaluate_model_on_validation(baseline_model, val_set, X_cols, y_cols)
    print(f"Baseline MAE: {baseline_mae:.4f}")

    # Setup di Optuna
    optuna.logging.set_verbosity(optuna.logging.INFO)
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=args.seed),
        study_name=f"{model_name}_glucose_prediction",
    )

    # Costruisci la funzione obiettivo
    objective_fn = create_objective_function(
        model_name, train_set, val_set, X_cols, y_cols, args.seed
    )

    # Esegui l'ottimizzazione
    print(f"Starting optimization with {args.n_trials} trials...")
    study.optimize(
        objective_fn,
        n_trials=args.n_trials,
        timeout=args.timeout,
        show_progress_bar=True,
    )

    # Salva lo studio con i migliori parametri ottenuti
    study_path = f"{args.study_dir}/{model_name}_optuna_study.pkl"
    with open(study_path, "wb") as f:
        pickle.dump(study, f)
    print(f"Optuna study saved to: {study_path}")

    return study, baseline_mae

In [ ]:
def train_and_evaluate_best_model(
    model_name, study, train_set, val_set, X_cols, y_cols, args
):
    """Addestra il modello con iperparametri ottimizzati e valutalo sul val_set"""
    print(
        f"\nTraining best {model_name.upper()} model and evaluating on validation set..."
    )

    best_params = study.best_params

    # Addestra il modello sul tran set
    best_model = create_model_instance(model_name, args.seed, best_params)
    best_model.fit(train_set[X_cols], train_set[y_cols[-1]])

    # Salva il modello
    model_path = f"{args.models_dir}/{model_name}_tuned.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)
    print(f"Best model saved to: {model_path}")

    # Genera le inferenze sul val set
    val_set_eval = val_set.copy()
    val_set_eval["y_pred"] = best_model.predict(val_set_eval[X_cols])
    val_set_eval = val_set_eval.rename(columns={y_cols[-1]: "target"})
    val_set_eval = rescale_data(val_set_eval, ["target", "y_pred"])

    # Seleziona le colonne di output
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = val_set_eval[output_columns]

    # Stampa e salva i risultati
    print(f"\n{model_name.upper()} Validation Set Results:")
    print_results(results)

    results_path = f"{args.output_dir}/{model_name}_tuned_output.csv"
    results.to_csv(results_path, index=False)
    print(f"Validation results saved to: {results_path}")

    return best_model, results

## Pipeline di tuning
Ottimizzazione sequenziale di XGBoost e LightGBM. Per ciascun modello: valutazione baseline, ricerca bayesiana degli iperparametri, addestramento del modello finale con i parametri ottimali.

In [ ]:
from lib.config import get_splits_dir

# Configura il tuning
class Args:
    def __init__(self):
        self.output_dir = "outputs/val_set"
        self.models_dir = "models/val_set"
        self.study_dir = "tuning/results"
        self.splits_dir = get_splits_dir()
        self.model = "both"  # Scegli "xgb", "lgb" oppure "both"
        self.n_trials = 40  # Numero di trial (valore usato nella tesi)
        self.timeout = 3600  # Timeout del tuning in secondi (1 ora)
        self.seed = 42


args = Args()

# Verifica i parametri
print(f"Configurazione:")
print(f"Model(s): {args.model.upper()}")
print(f"Trials: {args.n_trials}")
print(f"Timeout: {args.timeout}")
print(f"Seed: {args.seed}")
print(f"Split directory: {args.splits_dir}")
print(f"Study directory: {args.study_dir}")
print(f"Output directory: {args.output_dir}")
print(f"Models directory: {args.models_dir}")

In [ ]:
# Setup
os.makedirs(args.output_dir, exist_ok=True)
os.makedirs(args.models_dir, exist_ok=True)
os.makedirs(args.study_dir, exist_ok=True)
np.random.seed(args.seed)

In [ ]:
# Load data
train_set, val_set, test_set, X_cols, y_cols = load_splits(args.splits_dir)
print(f"Train set: {train_set.shape}")
print(f"Validation set: {val_set.shape}")
print(f"Test set: {test_set.shape}")
print(f"Features: {len(X_cols)}")
print(f"Target: {y_cols}")

In [ ]:
# Determina il modello da tunare
models_to_tune = ["xgb", "lgb"] if args.model == "both" else [args.model]

# Salva i risultati
all_results = {}

# Processa ciascun modello se tuni entrambi
for model_name in models_to_tune:
    try:
        # Esegui l'ottimizzazione
        study, baseline_mae = run_optimization(
            model_name, train_set, val_set, X_cols, y_cols, args
        )

        # Addestra il modello con i migliori parametri ottenuti e salvalo
        best_model, results = train_and_evaluate_best_model(
            model_name, study, train_set, val_set, X_cols, y_cols, args
        )

        # Salva i risultati
        all_results[model_name] = {
            "baseline_mae": baseline_mae,
            "best_mae": study.best_value,
            "improvement": baseline_mae - study.best_value,
            "improvement_pct": ((baseline_mae - study.best_value) / baseline_mae * 100),
            "best_params": study.best_params,
        }

        print(f"\n{model_name.upper()} optimization completed.")

    except Exception as e:
        print(f"\nError processing {model_name.upper()}: {str(e)}")
        continue

In [ ]:
# Final Summary
print(f"\n{'='*60}")
print("FINAL SUMMARY")
print(f"{'='*60}")

for model_name, results in all_results.items():
    print(f"\n{model_name.upper()}:")
    print(f"  Baseline MAE: {results['baseline_mae']:.4f}")
    print(f"  Best MAE: {results['best_mae']:.4f}")
    print(
        f"  Improvement: {results['improvement']:.4f} ({results['improvement_pct']:.2f}%)"
    )
    print(f"  Best parameters: {results['best_params']}")
    print(f"  Files: model, study, results saved to {args.output_dir}/")

print(f"\nAll optimization tasks completed.")